In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA_DIR = Path("data/modelling")
FEATURES_PATH = DATA_DIR / "features.parquet"
TARGET_PATH = DATA_DIR / "target.parquet"

# Parquet générés par le pipeline de préparation actuel.
SIMPLE_FEATURES = ["lag_1d", "lag_7d"]
ALL_FEATURES = [
    "lag_1d",
    "lag_7d",
    "lag_30d",
    "lag_365d",
    "rolling_mean_7d",
    "rolling_mean_30d",
]
TARGET = "consumption_kwh"
FEATURE_COLUMNS = sorted(set(SIMPLE_FEATURES + ALL_FEATURES))


In [ ]:
# On ne charge que les colonnes réellement nécessaires pour limiter la RAM.
feature_index = pq.read_table(
    FEATURES_PATH, columns=["individual", "timestamp"]
).to_pandas().index

df_feature = pq.read_table(
    FEATURES_PATH, columns=FEATURE_COLUMNS
).to_pandas().astype("float32")
df_feature.index = feature_index

target_index = pq.read_table(
    TARGET_PATH, columns=["individual", "timestamp"]
).to_pandas().index

df_target = pq.read_table(
    TARGET_PATH, columns=[TARGET]
).to_pandas().astype("float32")
df_target.index = target_index


In [ ]:
df_target

In [ ]:
df_feature

In [ ]:
# Fusion features + cible, puis libération immédiate des DataFrames intermédiaires.
df = df_feature.join(df_target, how="inner")
del df_feature, df_target


In [ ]:
df

In [ ]:
# On retire uniquement les lignes inutilisables pour les variables choisies.
df = df.dropna(subset=FEATURE_COLUMNS + [TARGET])


# Example: how to split df with multi indexes

In [ ]:
# Exemple : filtrer une année à partir du niveau 'timestamp' du MultiIndex.
df[df.index.get_level_values("timestamp").year == 2011]


In [ ]:
years = df.index.get_level_values("timestamp").year
df_train = df[years.isin([2011, 2012])].copy()


In [ ]:
df_test = df[years.isin([2013, 2014])].copy()
del df, years


In [ ]:
df_train.shape

In [ ]:
df_test.shape

In [ ]:
# Vérification des colonnes utilisées par chaque modèle
print("Ridge :", SIMPLE_FEATURES)
print("BayesianRidge :", ALL_FEATURES)
print("Cible :", TARGET)


In [ ]:
from sklearn import linear_model

In [ ]:
model = linear_model.Ridge(alpha=0.1)

In [ ]:
model.fit(df_train[SIMPLE_FEATURES], df_train[TARGET])

In [ ]:
model.coef_

In [ ]:
# Une ligne seule avec iloc[0] devient une Series 1D.
# Pour predict(), scikit-learn attend un tableau 2D : (n_samples, n_features).
sample = df_test.loc[:, SIMPLE_FEATURES].iloc[[0]]
sample


In [ ]:
# Prédiction sur UN échantillon en conservant sa forme 2D.
model.predict(sample)


In [ ]:
# Évaluation simple du modèle Ridge sur le jeu de test
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

ridge_pred = model.predict(df_test[SIMPLE_FEATURES])
ridge_mae = mean_absolute_error(df_test[TARGET], ridge_pred)
ridge_rmse = root_mean_squared_error(df_test[TARGET], ridge_pred)

print(f"Ridge MAE  : {ridge_mae:.3f} kWh")
print(f"Ridge RMSE : {ridge_rmse:.3f} kWh")


In [ ]:
bayesian_model = linear_model.BayesianRidge()

In [ ]:
bayesian_model.fit(df_train[ALL_FEATURES], df_train[TARGET])